> 在线讲义：[第 11 章 · 策略改进与策略迭代：评估一下，改一改，再评估](https://wikieden.github.io/rl-from-scratch/lessons/11-policy-iteration.html) · 本书《从零学强化学习》

# 第 11 章 · 策略改进与策略迭代：评估一下，改一改，再评估

本章回答：知道了一份策略的价值，怎么得到一份更好的策略？反复"评估 → 改进"，为什么一定会走到最优？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/11-policy-iteration.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 对随机策略贪心一次 | `greedy_policy(P, v_rand, 1.0)` | 第二步的箭头图 |
| 策略迭代（4×4） | `policy_iteration(P, 1.0, policy_evaluation)` | 3 轮，价值 0、−1、−2、−3 |
| 策略迭代（5×5） | `policy_iteration(G, 0.9, solve_v)` | 5 轮，$v_*(A) = 24.4$ |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `q_from_v`**（来自 `ch08_bellman.py`）：q_π(s, a) = Σ p(s', r|s, a) [r + γ v_π(s')]。

In [ ]:
def q_from_v(P, v, gamma, s, a):
    """q_π(s, a) = Σ p(s', r|s, a) [r + γ v_π(s')]。"""
    return sum(p * (r + (0 if done else gamma * v[ns])) for p, ns, r, done in P[s][a])

**函数 `greedy_policy`**（来自 `ch11_policy_iteration.py`）：每个状态在 q 最大的动作之间平分概率（并列时都留着，方便看）。

In [ ]:
def greedy_policy(P, v, gamma, eps=1e-9):
    """每个状态在 q 最大的动作之间平分概率（并列时都留着，方便看）。"""
    pi = {}
    for s in P:
        qs = {a: q_from_v(P, v, gamma, s, a) for a in P[s]}
        best = max(qs.values())
        top = [a for a, q in qs.items() if q > best - eps]
        pi[s] = {a: (1 / len(top) if a in top else 0.0) for a in P[s]}
    return pi

**常量 `MOVES`**（来自 `rl_envs.py`）

In [ ]:
MOVES = [(-1, 0), (1, 0), (0, 1), (0, -1)]

**函数 `gridworld_4x4`**（来自 `rl_envs.py`）：S&B Example 4.1：4×4，左上 (0) 和右下 (15) 是终点，每走一步 −1，撞墙原地不动。

In [ ]:
def gridworld_4x4():
    """S&B Example 4.1：4×4，左上 (0) 和右下 (15) 是终点，每走一步 −1，撞墙原地不动。"""
    n = 4
    P = {}
    for s in range(n * n):
        r, c = divmod(s, n)
        P[s] = {}
        for a, (dr, dc) in enumerate(MOVES):
            if s in (0, 15):
                P[s][a] = [(1.0, s, 0.0, True)]
                continue
            nr, nc = r + dr, c + dc
            ns = nr * n + nc if (0 <= nr < n and 0 <= nc < n) else s
            P[s][a] = [(1.0, ns, -1.0, ns in (0, 15))]
    return P

**函数 `gridworld_5x5`**（来自 `rl_envs.py`）：S&B Example 3.5（图 3.2）：5×5，A=(0,1) 任何动作 +10 跳到 A'=(4,1)，B=(0,3) +5 跳到 B'=(2,3)，

In [ ]:
def gridworld_5x5():
    """S&B Example 3.5（图 3.2）：5×5，A=(0,1) 任何动作 +10 跳到 A'=(4,1)，B=(0,3) +5 跳到 B'=(2,3)，
    撞墙原地不动 −1，其余 0。连续任务，不结束。状态编号 = 行 × 5 + 列。"""
    n = 5
    P = {}
    for r in range(n):
        for c in range(n):
            s = r * n + c
            P[s] = {}
            for a, (dr, dc) in enumerate(MOVES):
                if (r, c) == (0, 1):
                    P[s][a] = [(1.0, 4 * n + 1, 10.0, False)]
                elif (r, c) == (0, 3):
                    P[s][a] = [(1.0, 2 * n + 3, 5.0, False)]
                else:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < n and 0 <= nc < n:
                        P[s][a] = [(1.0, nr * n + nc, 0.0, False)]
                    else:
                        P[s][a] = [(1.0, s, -1.0, False)]
    return P

**函数 `sweep`**（来自 `ch10_policy_eval.py`）：一轮：每个状态都用旧的 v 算一遍右边（同步更新，书里叫 two-array 版本）。

In [ ]:
def sweep(P, pi, v, gamma):
    """一轮：每个状态都用旧的 v 算一遍右边（同步更新，书里叫 two-array 版本）。"""
    return np.array([sum(pa * q_from_v(P, v, gamma, s, a) for a, pa in pi[s].items()) for s in P])

**函数 `policy_evaluation`**（来自 `ch10_policy_eval.py`）：下面要用到的小工具。

In [ ]:
def policy_evaluation(P, pi, gamma=1.0, theta=1e-6, history=False):
    v, hist = np.zeros(len(P)), []
    while True:
        hist.append(v)
        new = sweep(P, pi, v, gamma)
        if np.max(np.abs(new - v)) < theta:
            return (new, hist) if history else new
        v = new

**函数 `uniform_policy`**（来自 `rl_envs.py`）：下面要用到的小工具。

In [ ]:
def uniform_policy(P):
    return {s: {a: 1.0 / len(P[s]) for a in P[s]} for s in P}

**函数 `policy_iteration`**（来自 `ch11_policy_iteration.py`）：下面要用到的小工具。

In [ ]:
def policy_iteration(P, gamma, evaluate):
    pi, rounds = uniform_policy(P), 0
    while True:
        v = evaluate(P, pi, gamma)
        new = greedy_policy(P, v, gamma)
        rounds += 1
        if new == pi:
            return pi, v, rounds
        pi = new

**常量 `MOVE_NAMES`**（来自 `rl_envs.py`）

In [ ]:
MOVE_NAMES = ["↑", "↓", "→", "←"]

**函数 `show_policy`**（来自 `ch11_policy_iteration.py`）：下面要用到的小工具。

In [ ]:
def show_policy(pi, n):
    rows = []
    for r in range(n):
        rows.append(" ".join("".join(MOVE_NAMES[a] for a, p in pi[r * n + c].items() if p > 0) or "·"
                             for c in range(n)))
    return "\n".join(rows)

**函数 `policy_matrices`**（来自 `ch08_bellman.py`）：把 (P, π) 压成 r_π[s] = 期望即时奖励、P_π[s, s'] = 一步转移概率（终点之后不再有价值）。

In [ ]:
def policy_matrices(P, pi):
    """把 (P, π) 压成 r_π[s] = 期望即时奖励、P_π[s, s'] = 一步转移概率（终点之后不再有价值）。"""
    n = len(P)
    r_pi, P_pi = np.zeros(n), np.zeros((n, n))
    for s in P:
        for a, pa in pi[s].items():
            for p, ns, r, done in P[s][a]:
                r_pi[s] += pa * p * r
                if not done:
                    P_pi[s, ns] += pa * p
    return r_pi, P_pi

**函数 `solve_v`**（来自 `ch08_bellman.py`）：v = r_π + γ P_π v  ⇒  (I − γ P_π) v = r_π。

In [ ]:
def solve_v(P, pi, gamma):
    """v = r_π + γ P_π v  ⇒  (I − γ P_π) v = r_π。"""
    r_pi, P_pi = policy_matrices(P, pi)
    return np.linalg.solve(np.eye(len(P)) - gamma * P_pi, r_pi)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
P = gridworld_4x4()

In [ ]:
v_rand = policy_evaluation(P, uniform_policy(P))

In [ ]:
pi1 = greedy_policy(P, v_rand, 1.0)

In [ ]:
print("4×4：对随机策略的价值贪心一次：\n" + show_policy(pi1, 4))

In [ ]:
pi, v, rounds = policy_iteration(P, 1.0, lambda P, pi, g: policy_evaluation(P, pi, g))

In [ ]:
print(f"策略迭代 {rounds} 轮停下，价值：\n{np.round(v.reshape(4, 4), 1)}")

In [ ]:
G = gridworld_5x5()

In [ ]:
pi5, v5, r5 = policy_iteration(G, 0.9, solve_v)

In [ ]:
print(f"5×5 网格策略迭代 {r5} 轮，v*(A) = {v5[1]:.1f}")

## 3. 代码里才看得到的

`evaluate` 是作为参数传进去的：4×4 用第 10 章的迭代评估，5×5 直接用第 8 章的 `solve_v` 解方程。策略迭代只要求"能算出 $v_\pi$"，用哪种办法都行。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. 5×5 网格用 γ = 0.5 做策略迭代：`policy_iteration(G, 0.5, solve_v)` 3 轮就停，$v_*(A) = 10.323$；再用 `show_policy` 打出箭头，看看远离 A 的格子箭头怎么变了。
2. γ = 0.99：5 轮，$v_*(A) = 204.04$。策略迭代的轮数几乎不随 γ 变。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 换不换看 q：q 比 v 大就换；
> 每格挑 q 最大，新策略处处不输（改进定理）；
> 评估、改进、再评估，策略不变就停，停下就是最优；
> 评估不必算到底，下一章只算一轮。